In [1]:
# ---------------------------------------------------------
# 1) COLAB ORTAMINI HAZIRLA
# ---------------------------------------------------------
# Bu notebook tek başına çalışır.
# GitHub'daki src klasörünü import etme.
# Tüm fonksiyonları burada tanımla.
# Böylece path/import hatasıyla uğraşma.
# ---------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# GPU varsa GPU kullan.
# T4 GPU seçtiysen burada "cuda" görmen lazım.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# Sonuçlar biraz daha benzer çıksın diye seed ver.
# Seed, rastgele başlangıçları kontrol etmeye çalışır.
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

Mounted at /content/drive
Using device: cuda
GPU: Tesla T4


In [2]:
# ---------------------------------------------------------
# 2) GENEL AYARLAR
# ---------------------------------------------------------
# Dataset yolu, model ayarları ve çıktı klasörleri burada.
# ---------------------------------------------------------

DATA_DIR = Path("/content/drive/MyDrive/turbofan-rul-data/CMAPSSData")

DATASETS = ["FD001", "FD002", "FD003", "FD004"]

# Çok büyük RUL değerlerini 125'e sabitle.
# Motor çok sağlıklıyken 250 mi kaldı 220 mi kaldı ayrımı çok kritik değil.
RUL_CAP = 125

# Deep learning modelleri son 30 cycle'a bakacak.
WINDOW_SIZE = 30

# Her batch'te 64 örnek kullan.
BATCH_SIZE = 64

# Her cycle'da 24 input var:
# 3 setting + 21 sensor
INPUT_SIZE = 24

# GRU/LSTM iç hafıza boyutu.
HIDDEN_SIZE = 64

# Final çalışma için 20 epoch.
# Daha iyi sonuç istersen sonra 30 yapıp tekrar çalıştırabilirsin.
EPOCHS = 20

# Adam optimizer learning rate.
LEARNING_RATE = 0.001


# Çıktıları Colab içinde buraya kaydet.
OUTPUT_DIR = Path("/content/turbofan_rul_outputs")

FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Data directory: /content/drive/MyDrive/turbofan-rul-data/CMAPSSData
Output directory: /content/turbofan_rul_outputs


In [3]:
# ---------------------------------------------------------
# 3) DATASET DOSYALARINI KONTROL ET
# ---------------------------------------------------------
# Eksik dosya varsa eğitime başlamadan yakala.
# ---------------------------------------------------------

missing_files = []

for dataset_id in DATASETS:
    expected_files = [
        DATA_DIR / f"train_{dataset_id}.txt",
        DATA_DIR / f"test_{dataset_id}.txt",
        DATA_DIR / f"RUL_{dataset_id}.txt",
    ]

    for file_path in expected_files:
        if not file_path.exists():
            missing_files.append(str(file_path))

if len(missing_files) == 0:
    print("All dataset files are available.")
else:
    print("Missing files:")
    for file_path in missing_files:
        print(file_path)

All dataset files are available.


In [4]:
# ---------------------------------------------------------
# 4) KOLONLARI TANIMLA VE DOSYA OKUMA FONKSİYONLARI YAZ
# ---------------------------------------------------------
# NASA C-MAPSS .txt dosyalarında kolon adı yok.
# Bu yüzden kolon isimlerini kendin ver.
# ---------------------------------------------------------

INDEX_COLUMNS = ["unit_id", "cycle"]

SETTING_COLUMNS = [
    "setting_1",
    "setting_2",
    "setting_3",
]

SENSOR_COLUMNS = [f"sensor_{i}" for i in range(1, 22)]

FEATURE_COLUMNS = SETTING_COLUMNS + SENSOR_COLUMNS

ALL_COLUMNS = INDEX_COLUMNS + SETTING_COLUMNS + SENSOR_COLUMNS


def load_train_data(data_dir, dataset_id):
    """
    Train dosyasını oku.
    Train verisi motorların failure'a kadar çalıştığı veridir.
    """

    file_path = data_dir / f"train_{dataset_id}.txt"

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        header=None,
        names=ALL_COLUMNS,
    )

    return df


def load_test_data(data_dir, dataset_id):
    """
    Test dosyasını oku.
    Test verisi motorlar failure olmadan önce kesilmiş veridir.
    """

    file_path = data_dir / f"test_{dataset_id}.txt"

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        header=None,
        names=ALL_COLUMNS,
    )

    return df


def load_test_rul(data_dir, dataset_id):
    """
    RUL cevap dosyasını oku.
    Bu dosya test motorlarının son gözlenen cycle'daki gerçek kalan ömrünü verir.
    """

    file_path = data_dir / f"RUL_{dataset_id}.txt"

    rul_df = pd.read_csv(
        file_path,
        sep=r"\s+",
        header=None,
        names=["final_RUL"],
    )

    # Dosyada unit_id yok.
    # Satır sırası motor numarasıdır.
    rul_df["unit_id"] = range(1, len(rul_df) + 1)

    return rul_df[["unit_id", "final_RUL"]]

In [5]:
# ---------------------------------------------------------
# 5) RUL ÜRETME FONKSİYONLARI
# ---------------------------------------------------------
# RUL = Remaining Useful Life
# Yani motorun arızaya kaç cycle kaldığı.
# ---------------------------------------------------------

def add_train_rul(train_df, cap=None):
    """
    Train verisine RUL ekle.

    Train verisinde motorlar failure'a kadar çalışır.
    Bu yüzden:
    RUL = max_cycle - current_cycle
    """

    df = train_df.copy()

    # Her motorun son cycle değerini bul.
    df["max_cycle"] = df.groupby("unit_id")["cycle"].transform("max")

    # Her satırda kalan ömrü hesapla.
    df["RUL"] = df["max_cycle"] - df["cycle"]

    # Büyük RUL değerlerini sınırla.
    if cap is not None:
        df["RUL"] = df["RUL"].clip(upper=cap)

    return df


def add_test_rul(test_df, test_rul_df, cap=None):
    """
    Test verisine RUL ekle.

    Test motorları failure'a kadar gitmez.
    RUL_FDxxx.txt dosyası son gözlenen cycle'daki kalan ömrü verir.
    """

    df = test_df.copy()

    # Testte her motorun gözlenen son cycle değerini bul.
    max_test_cycles = (
        df.groupby("unit_id")["cycle"]
        .max()
        .reset_index()
        .rename(columns={"cycle": "max_test_cycle"})
    )

    # max_test_cycle bilgisini test tablosuna ekle.
    df = df.merge(max_test_cycles, on="unit_id", how="left")

    # final_RUL cevap bilgisini test tablosuna ekle.
    df = df.merge(test_rul_df, on="unit_id", how="left")

    # Motorun gerçek failure cycle değerini bul.
    df["failure_cycle"] = df["max_test_cycle"] + df["final_RUL"]

    # Her cycle için kalan ömrü hesapla.
    df["RUL"] = df["failure_cycle"] - df["cycle"]

    # Büyük RUL değerlerini sınırla.
    if cap is not None:
        df["RUL"] = df["RUL"].clip(upper=cap)

    return df

In [6]:
# ---------------------------------------------------------
# 6) SLIDING WINDOW FONKSİYONLARI
# ---------------------------------------------------------
# Baseline model tek satıra bakıyordu:
# 1 cycle x 24 feature
#
# Deep learning model zaman penceresine bakacak:
# 30 cycle x 24 feature
#
# Örnek:
# cycle 1-30 -> target: cycle 30 RUL
# cycle 2-31 -> target: cycle 31 RUL
# ---------------------------------------------------------

def create_all_cycle_windows(df, feature_columns, window_size):
    """
    Her motor için tüm mümkün window'ları üret.
    Bu all-cycle evaluation içindir.
    """

    X_windows = []
    y_values = []

    for unit_id in df["unit_id"].unique():

        # Sadece tek motorun satırlarını al.
        engine_df = df[df["unit_id"] == unit_id].copy()

        # Cycle sırasını garantiye al.
        engine_df = engine_df.sort_values("cycle")

        engine_features = engine_df[feature_columns].values
        engine_rul = engine_df["RUL"].values

        # Motorun cycle sayısı window_size'dan küçükse atla.
        if len(engine_df) < window_size:
            continue

        # Sliding window üret.
        for start in range(0, len(engine_df) - window_size + 1):
            end = start + window_size

            X_window = engine_features[start:end]
            y_target = engine_rul[end - 1]

            X_windows.append(X_window)
            y_values.append(y_target)

    X = np.array(X_windows)
    y = np.array(y_values)

    return X, y


def create_last_cycle_windows(df, feature_columns, window_size):
    """
    Her motor için sadece son window'u üret.
    Bu last-cycle evaluation içindir.

    Test motorunun son 30 cycle'ını al.
    Target olarak son cycle'daki RUL değerini kullan.
    """

    X_windows = []
    y_values = []

    for unit_id in df["unit_id"].unique():

        engine_df = df[df["unit_id"] == unit_id].copy()
        engine_df = engine_df.sort_values("cycle")

        if len(engine_df) < window_size:
            continue

        last_window_df = engine_df.tail(window_size)

        X_window = last_window_df[feature_columns].values
        y_target = last_window_df["RUL"].values[-1]

        X_windows.append(X_window)
        y_values.append(y_target)

    X = np.array(X_windows)
    y = np.array(y_values)

    return X, y

In [7]:
# ---------------------------------------------------------
# 7) TEK DATASET HAZIRLAMA FONKSİYONU
# ---------------------------------------------------------
# Bu fonksiyon bir dataset için her şeyi hazırlar:
#
# 1. train/test/RUL dosyalarını oku
# 2. RUL ekle
# 3. scaling yap
# 4. all-cycle window üret
# 5. last-cycle window üret
# 6. PyTorch DataLoader oluştur
# ---------------------------------------------------------

def prepare_dataset(dataset_id):
    print("\n==============================")
    print(f"Preparing dataset: {dataset_id}")
    print("==============================")

    # 1) Dosyaları oku.
    train_raw = load_train_data(DATA_DIR, dataset_id)
    test_raw = load_test_data(DATA_DIR, dataset_id)
    test_rul = load_test_rul(DATA_DIR, dataset_id)

    print("Train raw shape:", train_raw.shape)
    print("Test raw shape:", test_raw.shape)

    # 2) RUL ekle.
    train_df = add_train_rul(train_raw, cap=RUL_CAP)
    test_df = add_test_rul(test_raw, test_rul, cap=RUL_CAP)

    print("Train with RUL shape:", train_df.shape)
    print("Test with RUL shape:", test_df.shape)

    # 3) Feature tablolarını al.
    X_train_df = train_df[FEATURE_COLUMNS]
    X_test_df = test_df[FEATURE_COLUMNS]

    # 4) Scaling yap.
    # Scaler sadece train veriye fit edilir.
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_df)
    X_test_scaled = scaler.transform(X_test_df)

    # 5) Scaled veriyi tekrar DataFrame yap.
    # unit_id, cycle, RUL window üretmek için lazım.
    train_scaled_df = pd.DataFrame(X_train_scaled, columns=FEATURE_COLUMNS)
    train_scaled_df["unit_id"] = train_df["unit_id"].values
    train_scaled_df["cycle"] = train_df["cycle"].values
    train_scaled_df["RUL"] = train_df["RUL"].values

    test_scaled_df = pd.DataFrame(X_test_scaled, columns=FEATURE_COLUMNS)
    test_scaled_df["unit_id"] = test_df["unit_id"].values
    test_scaled_df["cycle"] = test_df["cycle"].values
    test_scaled_df["RUL"] = test_df["RUL"].values

    # 6) Window üret.
    X_train_windows, y_train_windows = create_all_cycle_windows(
        train_scaled_df,
        FEATURE_COLUMNS,
        WINDOW_SIZE,
    )

    X_test_all_windows, y_test_all_windows = create_all_cycle_windows(
        test_scaled_df,
        FEATURE_COLUMNS,
        WINDOW_SIZE,
    )

    X_test_last_windows, y_test_last_windows = create_last_cycle_windows(
        test_scaled_df,
        FEATURE_COLUMNS,
        WINDOW_SIZE,
    )

    print("X_train_windows:", X_train_windows.shape)
    print("X_test_all_windows:", X_test_all_windows.shape)
    print("X_test_last_windows:", X_test_last_windows.shape)

    # 7) NumPy array'leri PyTorch tensor'a çevir.
    X_train_tensor = torch.tensor(X_train_windows, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_windows, dtype=torch.float32)

    X_test_all_tensor = torch.tensor(X_test_all_windows, dtype=torch.float32)
    y_test_all_tensor = torch.tensor(y_test_all_windows, dtype=torch.float32)

    X_test_last_tensor = torch.tensor(X_test_last_windows, dtype=torch.float32)
    y_test_last_tensor = torch.tensor(y_test_last_windows, dtype=torch.float32)

    # 8) Dataset oluştur.
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_all_dataset = TensorDataset(X_test_all_tensor, y_test_all_tensor)
    test_last_dataset = TensorDataset(X_test_last_tensor, y_test_last_tensor)

    # 9) DataLoader oluştur.
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    test_all_loader = DataLoader(
        test_all_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_last_loader = DataLoader(
        test_last_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    print("Train batches:", len(train_loader))
    print("Test all-cycle batches:", len(test_all_loader))
    print("Test last-cycle batches:", len(test_last_loader))

    return train_loader, test_all_loader, test_last_loader

In [8]:
# ---------------------------------------------------------
# 8) DEEP LEARNING MODELLERİ
# ---------------------------------------------------------
# Üç model kullan:
#
# GRU:
# Time-series için basit recurrent model.
#
# LSTM:
# Time-series için daha klasik recurrent model.
#
# 1D CNN:
# Zaman penceresi içinde local pattern yakalar.
# ---------------------------------------------------------

class GRURULModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape:
        # batch_size x window_size x feature_count

        gru_output, hidden = self.gru(x)

        # Son time step'i al.
        last_step = gru_output[:, -1, :]

        prediction = self.fc(last_step)
        prediction = prediction.squeeze(1)

        return prediction


class LSTMRULModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape:
        # batch_size x window_size x feature_count

        lstm_output, hidden = self.lstm(x)

        # Son time step'i al.
        last_step = lstm_output[:, -1, :]

        prediction = self.fc(last_step)
        prediction = prediction.squeeze(1)

        return prediction


class CNN1DRULModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        # Conv1d şu formatı ister:
        # batch_size x channel_count x sequence_length
        #
        # Bizim veri:
        # batch_size x window_size x feature_count
        #
        # Bu yüzden forward içinde transpose yap.
        self.conv1 = nn.Conv1d(
            in_channels=input_size,
            out_channels=64,
            kernel_size=3,
            padding=1,
        )

        self.relu = nn.ReLU()

        # Zaman boyutunu tek değere indir.
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        # x shape:
        # batch_size x window_size x feature_count

        # Conv1d için:
        # batch_size x feature_count x window_size
        x = x.transpose(1, 2)

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        # batch_size x 64 x 1 -> batch_size x 64
        x = x.squeeze(2)

        prediction = self.fc(x)
        prediction = prediction.squeeze(1)

        return prediction

In [9]:
# ---------------------------------------------------------
# 9) TRAIN / TEST / METRİK FONKSİYONLARI
# ---------------------------------------------------------

def train_one_epoch(model, train_loader, loss_function, optimizer, device):
    """
    Modeli 1 epoch eğit.
    """

    model.train()

    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Eski gradientleri sıfırla.
        optimizer.zero_grad()

        # Tahmin yap.
        predictions = model(batch_X)

        # Hata hesapla.
        loss = loss_function(predictions, batch_y)

        # Geri yayılım yap.
        loss.backward()

        # Ağırlıkları güncelle.
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    return average_loss


def evaluate_model(model, data_loader, loss_function, device):
    """
    Modeli test et.
    Bu fonksiyon ağırlık güncellemez.
    """

    model.eval()

    total_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            predictions = model(batch_X)

            loss = loss_function(predictions, batch_y)

            total_loss += loss.item()

            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(batch_y.cpu().numpy())

    average_loss = total_loss / len(data_loader)

    all_predictions = np.concatenate(all_predictions)
    all_targets = np.concatenate(all_targets)

    return average_loss, all_predictions, all_targets


def calculate_metrics(y_true, y_pred):
    """
    Regression metriklerini hesapla.
    """

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
    }


def train_model(
    dataset_id,
    model_name,
    model,
    train_loader,
    test_all_loader,
    test_last_loader,
):
    """
    Tek modeli eğit.
    Hem all-cycle hem last-cycle sonucunu hesapla.
    """

    print("\n------------------------------")
    print(f"Dataset: {dataset_id} | Model: {model_name}")
    print("------------------------------")

    model = model.to(device)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    train_losses = []
    test_all_losses = []

    for epoch in range(1, EPOCHS + 1):

        train_loss = train_one_epoch(
            model,
            train_loader,
            loss_function,
            optimizer,
            device,
        )

        test_all_loss, _, _ = evaluate_model(
            model,
            test_all_loader,
            loss_function,
            device,
        )

        train_losses.append(train_loss)
        test_all_losses.append(test_all_loss)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test All Loss: {test_all_loss:.4f}"
        )

    # Final all-cycle evaluation.
    all_loss, all_predictions, all_targets = evaluate_model(
        model,
        test_all_loader,
        loss_function,
        device,
    )

    all_metrics = calculate_metrics(all_targets, all_predictions)

    # Final last-cycle evaluation.
    last_loss, last_predictions, last_targets = evaluate_model(
        model,
        test_last_loader,
        loss_function,
        device,
    )

    last_metrics = calculate_metrics(last_targets, last_predictions)

    print("All-cycle metrics:", all_metrics)
    print("Last-cycle metrics:", last_metrics)

    result = {
        "dataset": dataset_id,
        "model_name": model_name,
        "model": model,
        "train_losses": train_losses,
        "test_all_losses": test_all_losses,
        "all_predictions": all_predictions,
        "all_targets": all_targets,
        "last_predictions": last_predictions,
        "last_targets": last_targets,
        "all_metrics": all_metrics,
        "last_metrics": last_metrics,
    }

    return result

In [10]:
# ---------------------------------------------------------
# 10) GRAFİK FONKSİYONLARI
# ---------------------------------------------------------

def safe_name(text):
    """
    Dosya adı için boşlukları alt çizgi yap.
    """
    return text.lower().replace(" ", "_")


def save_loss_plot(result, output_path):
    """
    Train loss ve test loss grafiği kaydet.
    """

    plt.figure(figsize=(8, 4))
    plt.plot(result["train_losses"], label="Train Loss")
    plt.plot(result["test_all_losses"], label="Test All-Cycle Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title(f'{result["dataset"]} - {result["model_name"]} Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def save_actual_vs_predicted(y_true, y_pred, title, output_path):
    """
    Gerçek RUL vs tahmin edilen RUL grafiği kaydet.
    """

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.4)

    min_value = min(y_true.min(), y_pred.min())
    max_value = max(y_true.max(), y_pred.max())

    # İdeal çizgi.
    # Noktalar bu çizgiye ne kadar yakınsa model o kadar iyi.
    plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

    plt.xlabel("Actual RUL")
    plt.ylabel("Predicted RUL")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def save_error_histogram(y_true, y_pred, title, output_path):
    """
    Prediction error histogramı kaydet.
    """

    errors = y_pred - y_true

    plt.figure(figsize=(8, 4))
    plt.hist(errors, bins=30)
    plt.xlabel("Prediction Error")
    plt.ylabel("Count")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

In [11]:
# ---------------------------------------------------------
# 11) TÜM DATASETLERİ VE TÜM MODELLERİ ÇALIŞTIR
# ---------------------------------------------------------
# 4 dataset x 3 model = 12 eğitim yapılır.
#
# Datasetler:
# FD001, FD002, FD003, FD004
#
# Modeller:
# GRU, LSTM, 1D CNN
# ---------------------------------------------------------

all_results = []
metric_rows = []

for dataset_id in DATASETS:

    train_loader, test_all_loader, test_last_loader = prepare_dataset(dataset_id)

    models = [
        ("GRU", GRURULModel(INPUT_SIZE, HIDDEN_SIZE)),
        ("LSTM", LSTMRULModel(INPUT_SIZE, HIDDEN_SIZE)),
        ("1D CNN", CNN1DRULModel(INPUT_SIZE)),
    ]

    for model_name, model in models:

        result = train_model(
            dataset_id=dataset_id,
            model_name=model_name,
            model=model,
            train_loader=train_loader,
            test_all_loader=test_all_loader,
            test_last_loader=test_last_loader,
        )

        all_results.append(result)

        # all-cycle satırı.
        metric_rows.append({
            "dataset": dataset_id,
            "model": model_name,
            "evaluation": f"all_cycles_window_{WINDOW_SIZE}",
            "window_size": WINDOW_SIZE,
            "hidden_size": HIDDEN_SIZE if model_name != "1D CNN" else None,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "MAE": result["all_metrics"]["MAE"],
            "RMSE": result["all_metrics"]["RMSE"],
            "R2": result["all_metrics"]["R2"],
        })

        # last-cycle satırı.
        metric_rows.append({
            "dataset": dataset_id,
            "model": model_name,
            "evaluation": f"last_cycle_window_{WINDOW_SIZE}",
            "window_size": WINDOW_SIZE,
            "hidden_size": HIDDEN_SIZE if model_name != "1D CNN" else None,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "MAE": result["last_metrics"]["MAE"],
            "RMSE": result["last_metrics"]["RMSE"],
            "R2": result["last_metrics"]["R2"],
        })

        # Klasörleri hazırla.
        dataset_figures_dir = FIGURES_DIR / dataset_id
        dataset_models_dir = MODELS_DIR / dataset_id

        dataset_figures_dir.mkdir(parents=True, exist_ok=True)
        dataset_models_dir.mkdir(parents=True, exist_ok=True)

        model_file_name = safe_name(model_name)

        # Loss grafiği.
        save_loss_plot(
            result,
            dataset_figures_dir / f"{model_file_name}_training_loss.png",
        )

        # All-cycle actual vs predicted.
        save_actual_vs_predicted(
            result["all_targets"],
            result["all_predictions"],
            f"{dataset_id} - {model_name} All-Cycle Actual vs Predicted",
            dataset_figures_dir / f"{model_file_name}_actual_vs_predicted_all_cycles.png",
        )

        # Last-cycle actual vs predicted.
        save_actual_vs_predicted(
            result["last_targets"],
            result["last_predictions"],
            f"{dataset_id} - {model_name} Last-Cycle Actual vs Predicted",
            dataset_figures_dir / f"{model_file_name}_actual_vs_predicted_last_cycle.png",
        )

        # All-cycle error histogram.
        save_error_histogram(
            result["all_targets"],
            result["all_predictions"],
            f"{dataset_id} - {model_name} All-Cycle Error Histogram",
            dataset_figures_dir / f"{model_file_name}_error_histogram_all_cycles.png",
        )

        # Last-cycle error histogram.
        save_error_histogram(
            result["last_targets"],
            result["last_predictions"],
            f"{dataset_id} - {model_name} Last-Cycle Error Histogram",
            dataset_figures_dir / f"{model_file_name}_error_histogram_last_cycle.png",
        )

        # Model ağırlığını kaydet.
        # GitHub'a koymak şart değil.
        torch.save(
            result["model"].state_dict(),
            dataset_models_dir / f"{model_file_name}_window{WINDOW_SIZE}.pt",
        )

        # Modeli CPU'ya al.
        # GPU belleğini temiz tut.
        result["model"] = result["model"].to("cpu")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


Preparing dataset: FD001
Train raw shape: (20631, 26)
Test raw shape: (13096, 26)
Train with RUL shape: (20631, 28)
Test with RUL shape: (13096, 30)
X_train_windows: (17731, 30, 24)
X_test_all_windows: (10196, 30, 24)
X_test_last_windows: (100, 30, 24)
Train batches: 278
Test all-cycle batches: 160
Test last-cycle batches: 2

------------------------------
Dataset: FD001 | Model: GRU
------------------------------
Epoch 01/20 | Train Loss: 6384.6464 | Test All Loss: 7498.6159
Epoch 02/20 | Train Loss: 4173.0670 | Test All Loss: 5159.7533
Epoch 03/20 | Train Loss: 2813.7914 | Test All Loss: 3536.2688
Epoch 04/20 | Train Loss: 1895.8370 | Test All Loss: 2409.9152
Epoch 05/20 | Train Loss: 1278.1671 | Test All Loss: 1629.2601
Epoch 06/20 | Train Loss: 866.5132 | Test All Loss: 1113.0478
Epoch 07/20 | Train Loss: 616.7007 | Test All Loss: 809.0035
Epoch 08/20 | Train Loss: 561.4607 | Test All Loss: 568.0244
Epoch 09/20 | Train Loss: 354.6151 | Test All Loss: 423.6055
Epoch 10/20 | Train L

In [12]:
# ---------------------------------------------------------
# 12) DEEP LEARNING SONUÇ TABLOSU
# ---------------------------------------------------------

deep_metrics_df = pd.DataFrame(metric_rows)

deep_metrics_df.to_csv(
    METRICS_DIR / "deep_learning_all_datasets_metrics.csv",
    index=False,
)

deep_metrics_df

,dataset,model,evaluation,window_size,hidden_size,epochs,learning_rate,MAE,RMSE,R2
0,FD001,GRU,all_cycles_window_30,30,64.0,20,0.001,10.241121,14.023966,0.777434
1,FD001,GRU,last_cycle_window_30,30,64.0,20,0.001,10.615510,14.127662,0.875712
2,FD001,LSTM,all_cycles_window_30,30,64.0,20,0.001,10.925615,15.540256,0.726704
3,FD001,LSTM,last_cycle_window_30,30,64.0,20,0.001,10.433534,14.198314,0.874465
4,FD001,1D CNN,all_cycles_window_30,30,NaN,20,0.001,16.127419,20.955456,0.503051
5,FD001,1D CNN,last_cycle_window_30,30,NaN,20,0.001,15.872536,21.903244,0.701250
6,FD002,GRU,all_cycles_window_30,30,64.0,20,0.001,16.537203,22.752903,0.461038
7,FD002,GRU,last_cycle_window_30,30,64.0,20,0.001,14.927616,20.607796,0.767760
8,FD002,LSTM,all_cycles_window_30,30,64.0,20,0.001,14.266849,20.031897,0.582238
9,FD002,LSTM,last_cycle_window_30,30,64.0,20,0.001,14.063708,19.323534,0.795804


In [13]:
# ---------------------------------------------------------
# 13) BASELINE + DEEP LEARNING KARŞILAŞTIRMASI
# ---------------------------------------------------------
# Baseline değerleri klasik ML pipeline'dan geldi.
# Hem all-cycle hem last-cycle baseline sonuçlarını ekle.
# ---------------------------------------------------------

baseline_rows = [
    # FD001 all-cycle
    ["FD001", "Linear Regression", "all_cycles", 1, None, None, None, 16.642204, 20.750290, 0.433911],
    ["FD001", "Random Forest", "all_cycles", 1, None, None, None, 12.446675, 17.158479, 0.612926],
    ["FD001", "Gradient Boosting", "all_cycles", 1, None, None, None, 12.617624, 17.346615, 0.604391],

    # FD001 last-cycle
    ["FD001", "Linear Regression", "last_cycle", 1, None, None, None, 16.568736, 20.830452, 0.729799],
    ["FD001", "Random Forest", "last_cycle", 1, None, None, None, 12.170970, 17.490075, 0.809509],
    ["FD001", "Gradient Boosting", "last_cycle", 1, None, None, None, 12.076587, 17.358160, 0.812372],

    # FD002 all-cycle
    ["FD002", "Linear Regression", "all_cycles", 1, None, None, None, 18.024789, 22.164596, 0.408811],
    ["FD002", "Random Forest", "all_cycles", 1, None, None, None, 15.905459, 19.635164, 0.536045],
    ["FD002", "Gradient Boosting", "all_cycles", 1, None, None, None, 17.395131, 20.763958, 0.481168],

    # FD002 last-cycle
    ["FD002", "Linear Regression", "last_cycle", 1, None, None, None, 17.529264, 21.364483, 0.752443],
    ["FD002", "Random Forest", "last_cycle", 1, None, None, None, 14.225466, 18.732593, 0.809679],
    ["FD002", "Gradient Boosting", "last_cycle", 1, None, None, None, 16.318892, 20.399135, 0.774309],

    # FD003 all-cycle
    ["FD003", "Linear Regression", "all_cycles", 1, None, None, None, 13.626351, 17.964969, 0.476811],
    ["FD003", "Random Forest", "all_cycles", 1, None, None, None, 9.252999, 14.367106, 0.665386],
    ["FD003", "Gradient Boosting", "all_cycles", 1, None, None, None, 9.607541, 14.585438, 0.655139],

    # FD003 last-cycle
    ["FD003", "Linear Regression", "last_cycle", 1, None, None, None, 16.912881, 21.163195, 0.708027],
    ["FD003", "Random Forest", "last_cycle", 1, None, None, None, 14.768275, 20.195261, 0.734124],
    ["FD003", "Gradient Boosting", "last_cycle", 1, None, None, None, 15.039409, 20.783641, 0.718406],

    # FD004 all-cycle
    ["FD004", "Linear Regression", "all_cycles", 1, None, None, None, 15.865903, 20.181945, 0.371834],
    ["FD004", "Random Forest", "all_cycles", 1, None, None, None, 12.396637, 17.331466, 0.536746],
    ["FD004", "Gradient Boosting", "all_cycles", 1, None, None, None, 13.903127, 18.465747, 0.474125],

    # FD004 last-cycle
    ["FD004", "Linear Regression", "last_cycle", 1, None, None, None, 19.958166, 24.471757, 0.675829],
    ["FD004", "Random Forest", "last_cycle", 1, None, None, None, 15.267723, 20.657117, 0.769015],
    ["FD004", "Gradient Boosting", "last_cycle", 1, None, None, None, 17.630885, 22.228152, 0.732545],
]

baseline_df = pd.DataFrame(
    baseline_rows,
    columns=[
        "dataset",
        "model",
        "evaluation",
        "window_size",
        "hidden_size",
        "epochs",
        "learning_rate",
        "MAE",
        "RMSE",
        "R2",
    ],
)

comparison_df = pd.concat(
    [baseline_df, deep_metrics_df],
    ignore_index=True,
)

comparison_df.to_csv(
    METRICS_DIR / "baseline_vs_deep_learning_all_datasets.csv",
    index=False,
)

comparison_df

/tmp/ipykernel_3533/3341223797.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  comparison_df = pd.concat(


,dataset,model,evaluation,window_size,hidden_size,epochs,learning_rate,MAE,RMSE,R2
0,FD001,Linear Regression,all_cycles,1,NaN,None,NaN,16.642204,20.750290,0.433911
1,FD001,Random Forest,all_cycles,1,NaN,None,NaN,12.446675,17.158479,0.612926
2,FD001,Gradient Boosting,all_cycles,1,NaN,None,NaN,12.617624,17.346615,0.604391
3,FD001,Linear Regression,last_cycle,1,NaN,None,NaN,16.568736,20.830452,0.729799
4,FD001,Random Forest,last_cycle,1,NaN,None,NaN,12.170970,17.490075,0.809509
5,FD001,Gradient Boosting,last_cycle,1,NaN,None,NaN,12.076587,17.358160,0.812372
6,FD002,Linear Regression,all_cycles,1,NaN,None,NaN,18.024789,22.164596,0.408811
7,FD002,Random Forest,all_cycles,1,NaN,None,NaN,15.905459,19.635164,0.536045
8,FD002,Gradient Boosting,all_cycles,1,NaN,None,NaN,17.395131,20.763958,0.481168
9,FD002,Linear Regression,last_cycle,1,NaN,None,NaN,17.529264,21.364483,0.752443


In [14]:
# ---------------------------------------------------------
# 14) EN İYİ DEEP LEARNING SONUÇLARINI BUL
# ---------------------------------------------------------
# RMSE düşükse model daha iyidir.
# Dataset başına en iyi all-cycle ve last-cycle deep learning modelini bul.
# ---------------------------------------------------------

deep_all_cycle = deep_metrics_df[
    deep_metrics_df["evaluation"] == f"all_cycles_window_{WINDOW_SIZE}"
].copy()

deep_last_cycle = deep_metrics_df[
    deep_metrics_df["evaluation"] == f"last_cycle_window_{WINDOW_SIZE}"
].copy()

best_deep_all_cycle = (
    deep_all_cycle
    .sort_values(["dataset", "RMSE"])
    .groupby("dataset")
    .head(1)
)

best_deep_last_cycle = (
    deep_last_cycle
    .sort_values(["dataset", "RMSE"])
    .groupby("dataset")
    .head(1)
)

best_deep_all_cycle.to_csv(
    METRICS_DIR / "best_deep_learning_all_cycle.csv",
    index=False,
)

best_deep_last_cycle.to_csv(
    METRICS_DIR / "best_deep_learning_last_cycle.csv",
    index=False,
)

print("Best deep learning models - all-cycle:")
display(best_deep_all_cycle)

print("Best deep learning models - last-cycle:")
display(best_deep_last_cycle)

Best deep learning models - all-cycle:


,dataset,model,evaluation,window_size,hidden_size,epochs,learning_rate,MAE,RMSE,R2
0,FD001,GRU,all_cycles_window_30,30,64.0,20,0.001,10.241121,14.023966,0.777434
8,FD002,LSTM,all_cycles_window_30,30,64.0,20,0.001,14.266849,20.031897,0.582238
12,FD003,GRU,all_cycles_window_30,30,64.0,20,0.001,7.520111,12.149110,0.791176
20,FD004,LSTM,all_cycles_window_30,30,64.0,20,0.001,11.421570,17.883406,0.571303


Best deep learning models - last-cycle:


,dataset,model,evaluation,window_size,hidden_size,epochs,learning_rate,MAE,RMSE,R2
1,FD001,GRU,last_cycle_window_30,30,64.0,20,0.001,10.615510,14.127662,0.875712
9,FD002,LSTM,last_cycle_window_30,30,64.0,20,0.001,14.063708,19.323534,0.795804
13,FD003,GRU,last_cycle_window_30,30,64.0,20,0.001,8.896507,12.965367,0.890415
21,FD004,LSTM,last_cycle_window_30,30,64.0,20,0.001,15.120835,20.631151,0.767099


In [15]:
# ---------------------------------------------------------
# 15) DATASET BAZLI RMSE KARŞILAŞTIRMA GRAFİKLERİ
# ---------------------------------------------------------
# Her dataset için:
# klasik ML modelleri + deep learning modelleri karşılaştır.
# ---------------------------------------------------------

for dataset_id in DATASETS:

    dataset_figures_dir = FIGURES_DIR / dataset_id
    dataset_figures_dir.mkdir(parents=True, exist_ok=True)

    # All-cycle karşılaştırması.
    all_cycle_df = comparison_df[
        (comparison_df["dataset"] == dataset_id)
        &
        (
            (comparison_df["evaluation"] == "all_cycles")
            |
            (comparison_df["evaluation"] == f"all_cycles_window_{WINDOW_SIZE}")
        )
    ].copy()

    plt.figure(figsize=(10, 4))
    plt.bar(all_cycle_df["model"], all_cycle_df["RMSE"])
    plt.xlabel("Model")
    plt.ylabel("RMSE")
    plt.title(f"{dataset_id} - All-Cycle Baseline vs Deep Learning RMSE")
    plt.xticks(rotation=20)
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(
        dataset_figures_dir / f"{dataset_id.lower()}_all_cycle_baseline_vs_deep_learning_rmse.png",
        dpi=150,
    )
    plt.close()

    # Last-cycle karşılaştırması.
    last_cycle_df = comparison_df[
        (comparison_df["dataset"] == dataset_id)
        &
        (
            (comparison_df["evaluation"] == "last_cycle")
            |
            (comparison_df["evaluation"] == f"last_cycle_window_{WINDOW_SIZE}")
        )
    ].copy()

    plt.figure(figsize=(10, 4))
    plt.bar(last_cycle_df["model"], last_cycle_df["RMSE"])
    plt.xlabel("Model")
    plt.ylabel("RMSE")
    plt.title(f"{dataset_id} - Last-Cycle Baseline vs Deep Learning RMSE")
    plt.xticks(rotation=20)
    plt.grid(axis="y")
    plt.tight_layout()
    plt.savefig(
        dataset_figures_dir / f"{dataset_id.lower()}_last_cycle_baseline_vs_deep_learning_rmse.png",
        dpi=150,
    )
    plt.close()

print("Comparison plots saved.")

Comparison plots saved.


In [16]:
# ---------------------------------------------------------
# 16) OLUŞAN DOSYALARI KONTROL ET
# ---------------------------------------------------------

print("Metrics files:")
!ls -lh /content/turbofan_rul_outputs/metrics

print("\nFigure files:")
!find /content/turbofan_rul_outputs/figures -maxdepth 2 -type f | head -80

print("\nModel files:")
!find /content/turbofan_rul_outputs/models -maxdepth 2 -type f

Metrics files:
total 20K
-rw-r--r-- 1 root root 4.2K May  6 20:07 baseline_vs_deep_learning_all_datasets.csv
-rw-r--r-- 1 root root  500 May  6 20:08 best_deep_learning_all_cycle.csv
-rw-r--r-- 1 root root  501 May  6 20:08 best_deep_learning_last_cycle.csv
-rw-r--r-- 1 root root 2.6K May  6 20:07 deep_learning_all_datasets_metrics.csv

Figure files:
/content/turbofan_rul_outputs/figures/FD003/1d_cnn_error_histogram_last_cycle.png
/content/turbofan_rul_outputs/figures/FD003/fd003_last_cycle_baseline_vs_deep_learning_rmse.png
/content/turbofan_rul_outputs/figures/FD003/gru_error_histogram_last_cycle.png
/content/turbofan_rul_outputs/figures/FD003/1d_cnn_error_histogram_all_cycles.png
/content/turbofan_rul_outputs/figures/FD003/1d_cnn_actual_vs_predicted_last_cycle.png
/content/turbofan_rul_outputs/figures/FD003/gru_training_loss.png
/content/turbofan_rul_outputs/figures/FD003/gru_actual_vs_predicted_last_cycle.png
/content/turbofan_rul_outputs/figures/FD003/1d_cnn_training_loss.png
/con

In [17]:
# ---------------------------------------------------------
# 17) TÜM OUTPUTS KLASÖRÜNÜ ZIP YAP
# ---------------------------------------------------------
# Bu zip'i indir.
# Sonra lokal projede outputs içine gerekli dosyaları koy.
# ---------------------------------------------------------

!zip -r /content/turbofan_rul_deep_learning_all_datasets_outputs.zip /content/turbofan_rul_outputs

print("Zip ready:")
print("/content/turbofan_rul_deep_learning_all_datasets_outputs.zip")

  adding: content/turbofan_rul_outputs/ (stored 0%)
  adding: content/turbofan_rul_outputs/metrics/ (stored 0%)
  adding: content/turbofan_rul_outputs/metrics/best_deep_learning_last_cycle.csv (deflated 47%)
  adding: content/turbofan_rul_outputs/metrics/deep_learning_all_datasets_metrics.csv (deflated 63%)
  adding: content/turbofan_rul_outputs/metrics/baseline_vs_deep_learning_all_datasets.csv (deflated 65%)
  adding: content/turbofan_rul_outputs/metrics/best_deep_learning_all_cycle.csv (deflated 47%)
  adding: content/turbofan_rul_outputs/figures/ (stored 0%)
  adding: content/turbofan_rul_outputs/figures/FD003/ (stored 0%)
  adding: content/turbofan_rul_outputs/figures/FD003/1d_cnn_error_histogram_last_cycle.png (deflated 24%)
  adding: content/turbofan_rul_outputs/figures/FD003/fd003_last_cycle_baseline_vs_deep_learning_rmse.png (deflated 18%)
  adding: content/turbofan_rul_outputs/figures/FD003/gru_error_histogram_last_cycle.png (deflated 22%)
  adding: content/turbofan_rul_outpu